In [1]:
EXP1 = "G6sulfur"     # subtrahend  (delta = EXP1 - EXP2)
EXP2 = "SSP245"

In [2]:
output_file = "/gws/ssde/j25b/impose/bidyut/analysis_transient_data/Chadwick_decomposition_G6sulfur_minus_SSP245.nc"
# output_file = "/gws/ssde/j25b/impose/bidyut/analysis_transient_data/Chadwick_decomposition_SSP585_minus_G6sulfur.nc"

In [3]:
import numpy as np
import xarray as xr
import cftime
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import xarray as xr
from pathlib import Path

import sys
import os

analysis_path = os.path.abspath("../20260112_Basic_Analysis")
sys.path.append(analysis_path)

import myfunctions as mf

#Package to suppress Python warnings
import warnings
warnings.filterwarnings("ignore")


import xesmf as xe
import xarray as xr
import numpy as np

# =========================
# Base CEDA paths
# =========================

CEDA_BASE = Path("/badc/cmip6/data/CMIP6")


#Model Names
MODELS = {
    "UKESM1-0-LL":  {"institution": "MOHC",         "ensemble": "r1i1p1f2",  "grid": "gn",},
    "CNRM-ESM2-1":  {"institution": "CNRM-CERFACS", "ensemble": "r1i1p1f2",  "grid": "gr",},
    "MPI-ESM1-2-LR":{"institution": "MPI-M",        "ensemble": "r1i1p1f1",  "grid": "gn",},
    "CESM2-WACCM":  {"institution": "NCAR",         "ensemble": "r1i1p1f1",  "grid": "gn",},
    "IPSL-CM6A-LR": {"institution": "IPSL",         "ensemble": "r1i1p1f1",  "grid": "gr",},
}


# =========================
# Experiment registry
# =========================

EXP_REGISTRY = {
    "piControl": {"project": "CMIP",        "scenario": "piControl"},
    "HIST":      {"project": "CMIP",        "scenario": "historical"},
    "SSP245":    {"project": "ScenarioMIP", "scenario": "ssp245"},
    "SSP585":    {"project": "ScenarioMIP", "scenario": "ssp585"},
    "G6solar":   {"project": "GeoMIP",      "scenario": "G6solar"},
    "G6sulfur":  {"project": "GeoMIP",      "scenario": "G6sulfur"},
}

# Built automatically from EXP1 and EXP2
EXPERIMENTS = {
    EXP1: EXP_REGISTRY[EXP1],
    EXP2: EXP_REGISTRY[EXP2],
}


In [4]:
# ================================================================
# Utility: year-range label per experiment
# ================================================================

def _exp_year_label(exp_name):
    """Return the year-range suffix for an experiment name."""
    if exp_name in ("G6sulfur", "SSP585", "SSP245"):
        return "20212100"
    elif exp_name == "HIST":
        return "19212000"
    else:
        raise ValueError(f"Unknown experiment '{exp_name}'. "
                         "Add it to _exp_year_label().")

def _time_slice(exp_name):
    """Return the (start, end) time slice strings for an experiment."""
    if exp_name in ("G6sulfur", "SSP585", "SSP245"):
        return ("2021-01-01", "2100-12")   # 80 years
    elif exp_name == "HIST":
        return ("1921-01-01", "2000-12")   # 80 years
    else:
        raise ValueError(f"Unknown experiment '{exp_name}'.")


def roll_lon(da):
    """Roll longitude from 0-360 to -180-180."""
    da = da.assign_coords(lon=(((da.lon + 180) % 360) - 180))
    return da.sortby('lon')


def regrid_to_common(da, target_lat=None, target_lon=None):
    """Regrid a DataArray to common lat/lon grid."""
    if target_lat is None:
        target_lat = np.arange(-50, 51, 2.5)
    elif isinstance(target_lat, tuple):
        target_lat = np.arange(target_lat[0], target_lat[1]+1, 2.5)
    elif isinstance(target_lat, (int, float)):
        target_lat = np.arange(-target_lat, target_lat+1, 2.5)

    if target_lon is None:
        target_lon = np.arange(0, 360, 2.5)  # always default to full lon
    elif isinstance(target_lon, tuple):
        target_lon = np.arange(target_lon[0], target_lon[1]+1, 2.5)

    ds_out = xr.Dataset({
        'lat': (['lat'], target_lat),
        'lon': (['lon'], target_lon)
    })

    # Handle both DataSet and DataArray cases
    if isinstance(da, xr.Dataset):
        ds_in = da
    else:
        ds_in = da.to_dataset(name='data')
        
    regridder = xe.Regridder(ds_in, ds_out, method='bilinear', reuse_weights=False)
    return regridder(da) 
    
def climatology_and_uncertainty(da_year, block_size=30):
    """
    da_year: DataArray with dimension 'year'
    Returns:
        clim_mean  : mean climatology (mean of 30-yr block means)
        clim_sd    : std dev across 30-yr block means
        block_means: DataArray of each 30-yr mean
    """

    n_years = da_year.sizes["year"]
    n_blocks = n_years // block_size

    # Trim excess years
    da_trim = da_year.isel(year=slice(0, n_blocks * block_size))

    # Create block index
    block = xr.DataArray(
        np.repeat(np.arange(n_blocks), block_size),
        dims="year",
        coords={"year": da_trim.year},
        name="block"
    )

    # Compute 30-year means
    block_means = (
        da_trim
        .groupby(block)
        .mean(dim="year")*86400
    )

    # Climatological mean (mean of 30-year means)
    clim_mean = block_means.mean(dim="block")

    # Spread across 30-year climatologies
    clim_sd = block_means.std(dim="block")

    return clim_mean, clim_sd, block_means


def load_model_data(base_path, model_name, scenario, mf):
    """
    Generic loader for model data based on model and scenario.

    Parameters
    ----------
    base_path : str or Path
        File path(s) to load
    model_name : str
    scenario : str
    mf : module
        Your module with open_files functions

    Returns
    -------
    xarray.Dataset
    """
    print(str(base_path))

    if model_name == "CESM2-WACCM":
        if scenario == "G6sulfur":
            return mf.open_files_CESM_G6sulfur(base_path)
        elif scenario == "ssp585":
            return mf.open_files_CESM_ssp585(base_path)
        else:
            return mf.open_files(str(base_path))

    elif model_name == "IPSL-CM6A-LR":
        if scenario == "ssp585":
            return mf.open_files_IPSL_ssp585(base_path)
        else:
            return mf.open_files(str(base_path))

    else:
        return mf.open_files(str(base_path))



def load_model_data_alt_path(base_path, project, institution, model_name, scenario, ensemble, grid, mf, var_name=None):
    """
    Generic loader for model data based on model and scenario.
    
    Parameters
    ----------
    base_path : str or Path
        File path(s) to load
    model_name : str
    scenario : str
    mf : module
        Your module with open_files functions
    var_name : str, optional
        Variable name (e.g., 'ua', 'va') to check for downloaded files
    
    Returns
    -------
    xarray.Dataset
    """
    print(str(base_path))
    
    # Check if files exist at base_path, otherwise look in downloaded directory
    base_path = Path(base_path)
    downloaded_base = Path("/gws/ssde/j25b/impose/bidyut/data")
    
    # if not base_path.exists() or not list(base_path.glob("*.nc")):
    if not any(base_path.glob("*.nc")):
        # Try downloaded directory
        if var_name:
            # --- special-case ensemble override ---
            if model_name == "CESM2-WACCM":
                ensemble = "r1i1p1f2" if scenario == "G6sulfur" else "r1i1p1f1"
            alt_path = (downloaded_base / project / institution / model_name / scenario / ensemble / "Amon" / 
                       var_name / grid / "latest")
            print(f"Alt path is: {alt_path}")
            if alt_path.exists():
                print(f"  → Using downloaded files at: {alt_path}")
                base_path = alt_path
    
    if model_name == "CESM2-WACCM":
        if scenario == "G6sulfur":
            return mf.open_files_CESM_G6sulfur(base_path)
        elif scenario == "ssp585":
            return mf.open_files_CESM_ssp585(base_path)
        else:
            return mf.open_files(str(base_path))
    elif model_name == "IPSL-CM6A-LR":
        if scenario == "ssp585":
            return mf.open_files_IPSL_ssp585(base_path)
        else:
            return mf.open_files(str(base_path))
    else:
        return mf.open_files(str(base_path))


def roll_and_regrid(varname):
    return roll_lon(regrid_to_common(varname, target_lat=50))



# CHADWICK DECOMPOSITION

Tropical mean $\omega_{{500}}$: $\alpha = \Delta\overline{{\omega}}_{{500}}\ /\ \overline{{\omega}}_{{500}}^{{\mathrm{{SSP245}}}}$


https://claude.ai/chat/f4f22eb0-7caa-458d-a248-36be4443e352

In [5]:
# # ================================================================
# # COMPUTE ALPHA FOR CHADWICK DECOMPOSITION
# # alpha = timemean(tropmean(dM*)) / timemean(tropmean(M*_ref))
# # tropical mean: 30S-30N, area-weighted
# # ================================================================

# alpha_store = {}  # store per-model for inspection

# for model_name, model_meta in MODELS.items():

#     omega_store_model = {}

#     for exp, meta in EXPERIMENTS.items():

#         if model_name == "CESM2-WACCM":
#             ensemble = "r1i1p1f2" if meta["scenario"] == "G6sulfur" else "r1i1p1f1"
#         else:
#             ensemble = model_meta["ensemble"]

#         def make_base(vname):
#             return (
#                 CEDA_BASE
#                 / meta["project"]
#                 / model_meta["institution"]
#                 / model_name
#                 / meta["scenario"]
#                 / ensemble
#                 / "Amon"
#                 / vname
#                 / model_meta["grid"]
#                 / "latest"
#             )

#         # Load pr and huss
#         for vname in ["pr", "huss"]:
#             base   = make_base(vname)
#             ds     = load_model_data_alt_path(base, meta["project"],
#                                               model_meta["institution"],
#                                               model_name, meta["scenario"],
#                                               ensemble, model_meta["grid"], mf, vname)
#             t_start, t_end = _time_slice(exp)
#             ds = ds.sel(time=slice(t_start, t_end))
#             ds = roll_and_regrid(ds)
#             omega_store_model.setdefault(exp, {})[vname] = ds[vname]

#         print(f"  [{model_name}] {exp}: pr and huss loaded")

#     # --- Compute M* for each experiment ---
#     pr_ssp   = omega_store_model[EXP2]['pr']
#     huss_ssp = omega_store_model[EXP2]['huss']
#     pr_g6    = omega_store_model[EXP1]['pr']
#     huss_g6  = omega_store_model[EXP1]['huss']

#     # Align time
#     common_time = np.intersect1d(pr_ssp.time.values, pr_g6.time.values)
#     pr_ssp   = pr_ssp.sel(time=common_time)
#     huss_ssp = huss_ssp.sel(time=common_time)
#     pr_g6    = pr_g6.sel(time=common_time)
#     huss_g6  = huss_g6.sel(time=common_time)

#     # M* = P / q_s
#     huss_ssp_safe = huss_ssp.where(huss_ssp > 1e-6, 1e-6)
#     huss_g6_safe  = huss_g6.where(huss_g6  > 1e-6, 1e-6)
#     Mstar_ref = pr_ssp / huss_ssp_safe    # (time, lat, lon)
#     Mstar_g6  = pr_g6  / huss_g6_safe    # (time, lat, lon)
#     dMstar    = Mstar_g6 - Mstar_ref      # (time, lat, lon)

#     # --- Tropical mean (area-weighted, 30S-30N) ---
#     Mstar_ref_trop = Mstar_ref.sel(lat=slice(-30, 30))
#     dMstar_trop    = dMstar.sel(lat=slice(-30, 30))

#     weights = np.cos(np.deg2rad(Mstar_ref_trop.lat))

#     # Time series of tropical mean (one value per month)
#     # Time series of tropical mean (one value per month)
#     Mstar_ref_tropmean_ts = Mstar_ref_trop.weighted(weights).mean(dim=['lat', 'lon'])  # (time,)
#     dMstar_tropmean_ts    = dMstar_trop.weighted(weights).mean(dim=['lat', 'lon'])      # (time,)

#     # Resample to annual means
#     Mstar_ref_tropmean_ts = Mstar_ref_tropmean_ts.resample(time='YE').mean()   # (n_years,)
#     dMstar_tropmean_ts    = dMstar_tropmean_ts.resample(time='YE').mean()      # (n_years,)

#     # Scalar time means (now mean of annual means)
#     Mstar_ref_tropmean = float(Mstar_ref_tropmean_ts.mean(dim='time'))
#     dMstar_tropmean    = float(dMstar_tropmean_ts.mean(dim='time'))

#     alpha = dMstar_tropmean / Mstar_ref_tropmean

#     alpha_store[model_name] = {
#         'Mstar_ref_tropmean_ts': Mstar_ref_tropmean_ts,   # (time,) — for plotting
#         'dMstar_tropmean_ts':    dMstar_tropmean_ts,       # (time,) — for plotting
#         'Mstar_ref_tropmean':    Mstar_ref_tropmean,       # scalar
#         'dMstar_tropmean':       dMstar_tropmean,          # scalar
#         'alpha':                 alpha,                    # scalar
#     }

#     print(f"  [{model_name}]  M*_ref tropmean = {Mstar_ref_tropmean:.6f}"
#           f"  dM* tropmean = {dMstar_tropmean:.6f}"
#           f"  alpha = {alpha:.4f}")

# # --- MMM alpha ---
# alpha_mmm = np.mean([v['alpha'] for v in alpha_store.values()])
# print(f"\nMMM alpha = {alpha_mmm:.4f}")
# print("\nPer-model alpha:")
# for m, v in alpha_store.items():
#     print(f"  {m}: {v['alpha']:.4f}")

In [6]:
# # ================================================================
# # PLOT: time series of tropmean(dM*) vs alpha * tropmean(M*_ref)
# # to verify alpha is a good approximation
# # ================================================================

# fig, axes = plt.subplots(2, 3, figsize=(15, 10))
# axes = axes.flatten()

# for idx, (model_name, res) in enumerate(alpha_store.items()):
#     ax    = axes[idx]
#     alpha = res['alpha']

#     ts_Mstar = res['Mstar_ref_tropmean_ts']
#     ts_dMstar = res['dMstar_tropmean_ts']
#     ts_approx = alpha * ts_Mstar    # alpha * M*_ref should approximate dM*

#     time = np.arange(len(ts_dMstar))

#     ax.plot(time, ts_dMstar.values,
#             color='black', linewidth=1.5, label=r'$\Delta M^*$ (actual)')
#     ax.plot(time, ts_approx.values,
#             color='red',   linewidth=1.5, linestyle='--',
#             label=rf'$\alpha \cdot M^*_{{ref}}$  (α={alpha:.3f})')

#     ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
#     ax.set_xlabel('Time (months)', fontsize=10)
#     ax.set_ylabel(r'Tropical mean $M^*$ [kg m$^{-2}$ s$^{-1}$]', fontsize=10)
#     ax.set_title(model_name, fontsize=11, fontweight='bold')
#     ax.legend(fontsize=9, framealpha=0.9)
#     ax.grid(True, alpha=0.3)

# for idx in range(len(alpha_store), 6):
#     axes[idx].axis('off')

# fig.suptitle(r'Verification: $\Delta M^*_{{trop}}$ vs $\alpha \cdot M^*_{{ref,trop}}$'
#              f'\nMMM α = {alpha_mmm:.4f}',
#              fontsize=13, fontweight='bold')

# plt.tight_layout()
# plt.savefig('alpha_verification_timeseries.png', dpi=300, bbox_inches='tight')
# plt.show()

In [7]:
# # ================================================================
# # PLOT: time series of tropmean(dM*) vs alpha * tropmean(M*_ref)
# # to verify alpha is a good approximation
# # ================================================================

# fig, axes = plt.subplots(2, 3, figsize=(15, 10))
# axes = axes.flatten()

# for idx, (model_name, res) in enumerate(alpha_store.items()):
#     ax    = axes[idx]
#     alpha = res['alpha']

#     ts_Mstar = res['Mstar_ref_tropmean_ts']
#     ts_dMstar = res['dMstar_tropmean_ts']
#     ts_approx = alpha * ts_Mstar    # alpha * M*_ref should approximate dM*

#     time = np.arange(len(ts_dMstar))

#     # ax.plot(time, ts_dMstar.values, ts_Mstar.values,
#     #         color='black', linewidth=1.5, label=r'$\Delta M^*$ (actual)')
#     ax.scatter(ts_Mstar.values, ts_dMstar.values,
#            color='black', s=20, label=r'$\Delta M^*$ (actual)')
#     # ax.plot(ts_dMstar.values, ts_approx.values,
#     #         color='red',   linewidth=1.5, linestyle='--',
#     #         label=rf'$\alpha \cdot M^*_{{ref}}$  (α={alpha:.3f})')


#     x_line = np.linspace(ts_Mstar.values.min(), ts_Mstar.values.max(), 100)
#     ax.plot(x_line, alpha * x_line,
#             color='red', linewidth=1.5, linestyle='--',
#             label=rf'$\alpha \cdot M^*_{{ref}}$  (α={alpha:.3f})')
    
#     ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
#     ax.set_xlabel('Time (months)', fontsize=10)
#     ax.set_ylabel(r'Tropical mean $M^*$ [kg m$^{-2}$ s$^{-1}$]', fontsize=10)
#     ax.set_title(model_name, fontsize=11, fontweight='bold')
#     ax.legend(fontsize=9, framealpha=0.9)
#     ax.grid(True, alpha=0.3)

# for idx in range(len(alpha_store), 6):
#     axes[idx].axis('off')

# fig.suptitle(r'Verification: $\Delta M^*_{{trop}}$ vs $\alpha \cdot M^*_{{ref,trop}}$'
#              f'\nMMM α = {alpha_mmm:.4f}',
#              fontsize=13, fontweight='bold')

# plt.tight_layout()
# plt.savefig('alpha_scatter.png', dpi=150, bbox_inches='tight')
# plt.show()

In [8]:
# fig, axes = plt.subplots(2, 3, figsize=(15, 10))
# axes = axes.flatten()

# for idx, (model_name, res) in enumerate(alpha_store.items()):
#     ax     = axes[idx]
#     alpha  = res['alpha']

#     ts_Mstar  = res['Mstar_ref_tropmean_ts']
#     ts_dMstar = res['dMstar_tropmean_ts']

#     x = ts_Mstar.values
#     y = ts_dMstar.values

#     # --- Scatter ---
#     ax.scatter(x, y, color='black', s=25, zorder=3,
#                label=r'Annual mean')

#     # --- Alpha line (through origin) ---
#     x_line = np.linspace(x.min(), x.max(), 100)
#     ax.plot(x_line, alpha * x_line,
#             color='red', linewidth=2, linestyle='--',
#             label=rf'$\alpha \cdot M^*_{{ref}}$  (α={alpha:.4f})')

#     # --- OLS fit for comparison ---
#     from scipy import stats
#     slope, intercept, r_value, _, _ = stats.linregress(x, y)
#     ax.plot(x_line, slope * x_line + intercept,
#             color='steelblue', linewidth=1.5, linestyle='-',
#             label=rf'OLS (slope={slope:.4f}, R²={r_value**2:.2f})')

#     ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
#     ax.axvline(x.mean(), color='gray', linestyle=':', linewidth=0.8)

#     ax.set_xlabel(r'Trop. mean $M^*_{\rm ref}$ (SSP245) [kg m$^{-2}$ s$^{-1}$]', fontsize=10)
#     ax.set_ylabel(r'Trop. mean $\Delta M^*$ (G6$-$SSP245) [kg m$^{-2}$ s$^{-1}$]', fontsize=10)
#     ax.set_title(model_name, fontsize=11, fontweight='bold')
#     ax.legend(fontsize=8, framealpha=0.9)
#     ax.grid(True, alpha=0.3)

# for idx in range(len(alpha_store), 6):
#     axes[idx].axis('off')

# alpha_mmm = np.mean([v['alpha'] for v in alpha_store.values()])
# fig.suptitle(r'Verification: trop. mean $\Delta M^*$ vs $\alpha \cdot M^*_{\rm ref}$'
#              f'\nMMM α = {alpha_mmm:.4f}',
#              fontsize=13, fontweight='bold')

# plt.tight_layout()
# plt.savefig('alpha_scatter.png', dpi=150, bbox_inches='tight')
# plt.show()

In [9]:
# ================================================================
# CHADWICK ALPHA COMPUTATION
# M* = P / q_s  (time mean first, then spatial operations)
# alpha = tropmean(dM*) / tropmean(M*_SSP_timemean)
# ================================================================

alpha_store = {}

for model_name, model_meta in MODELS.items():

    raw = {}   # raw[exp]['pr'], raw[exp]['huss']

    for exp, meta in EXPERIMENTS.items():

        if model_name == "CESM2-WACCM":
            ensemble = "r1i1p1f2" if meta["scenario"] == "G6sulfur" else "r1i1p1f1"
        else:
            ensemble = model_meta["ensemble"]

        def make_base(vname):
            return (
                CEDA_BASE
                / meta["project"]
                / model_meta["institution"]
                / model_name
                / meta["scenario"]
                / ensemble
                / "Amon"
                / vname
                / model_meta["grid"]
                / "latest"
            )

        raw.setdefault(exp, {})

        for vname in ["pr", "huss"]:
            base = make_base(vname)
            ds   = load_model_data_alt_path(base, meta["project"],
                                            model_meta["institution"],
                                            model_name, meta["scenario"],
                                            ensemble, model_meta["grid"], mf, vname)
            t_start, t_end = _time_slice(exp)
            ds = ds.sel(time=slice(t_start, t_end))
            ds = roll_and_regrid(ds)
            raw[exp][vname] = ds[vname]   # (time, lat, lon)

        print(f"  [{model_name}] {exp}: pr and huss loaded")

    # ----------------------------
    # 1) TIME MEAN FIRST → (lat, lon)
    # ----------------------------
    pr_ssp_tm   = raw[EXP2]['pr'].mean(dim='time')     # (lat, lon)
    huss_ssp_tm = raw[EXP2]['huss'].mean(dim='time')   # (lat, lon)
    pr_g6_tm    = raw[EXP1]['pr'].mean(dim='time')     # (lat, lon)
    huss_g6_tm  = raw[EXP1]['huss'].mean(dim='time')   # (lat, lon)

    # ----------------------------
    # 2) M* = P / q_s  on time-mean fields → (lat, lon)
    # ----------------------------
    huss_ssp_safe = huss_ssp_tm.where(huss_ssp_tm > 1e-6, 1e-6)
    huss_g6_safe  = huss_g6_tm.where(huss_g6_tm   > 1e-6, 1e-6)

    Mstar_ssp = pr_ssp_tm / huss_ssp_safe   # M*_SSP  (lat, lon)
    Mstar_g6  = pr_g6_tm  / huss_g6_safe    # M*_SAI  (lat, lon)

    # ----------------------------
    # 3) dM* = M*_SAI - M*_SSP  → (lat, lon)
    # ----------------------------
    dMstar = Mstar_g6 - Mstar_ssp            # (lat, lon)

    # ----------------------------
    # 4) Tropical mean (area-weighted, 30S-30N) → scalars for alpha
    # ----------------------------
    Mstar_ssp_trop = Mstar_ssp.sel(lat=slice(-30, 30))
    dMstar_trop    = dMstar.sel(lat=slice(-30, 30))

    weights = np.cos(np.deg2rad(Mstar_ssp_trop.lat))

    tropmean_Mstar_ssp = float(Mstar_ssp_trop.weighted(weights).mean(dim=['lat', 'lon']))
    tropmean_dMstar    = float(dMstar_trop.weighted(weights).mean(dim=['lat', 'lon']))

    alpha = tropmean_dMstar / tropmean_Mstar_ssp

    alpha_store[model_name] = {
        'Mstar_ssp':          Mstar_ssp,           # (lat, lon) — for scatter x
        'Mstar_g6':           Mstar_g6,            # (lat, lon)
        'dMstar':             dMstar,              # (lat, lon) — for scatter y
        'tropmean_Mstar_ssp': tropmean_Mstar_ssp,  # scalar
        'tropmean_dMstar':    tropmean_dMstar,      # scalar
        'alpha':              alpha,                # scalar
    }

    print(f"  [{model_name}]  tropmean M*_SSP={tropmean_Mstar_ssp:.6f}  "
          f"tropmean dM*={tropmean_dMstar:.6f}  alpha={alpha:.4f}")

alpha_mmm = np.mean([v['alpha'] for v in alpha_store.values()])
print(f"\nMMM alpha = {alpha_mmm:.4f}")
for m, v in alpha_store.items():
    print(f"  {m}: alpha={v['alpha']:.4f}")

# ================================================================
# PLOT: scatter of dM* vs M*_SSP (lat, lon grid points)
# ================================================================

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (model_name, res) in enumerate(alpha_store.items()):
    ax    = axes[idx]
    alpha = res['alpha']

    x = res['Mstar_ssp'].values.flatten()   # M*_SSP timemean (lat, lon)
    y = res['dMstar'].values.flatten()      # dM* (lat, lon)

    valid   = np.isfinite(x) & np.isfinite(y)
    x_clean = x[valid]
    y_clean = y[valid]

    ax.scatter(x_clean, y_clean, s=4, alpha=0.3, color='black')

    # Alpha line through origin
    x_line = np.linspace(x_clean.min(), x_clean.max(), 200)
    ax.plot(x_line, alpha * x_line,
            color='red', linewidth=2, linestyle='--',
            label=rf'$\alpha={alpha:.4f}$')

    ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
    ax.axvline(0, color='gray', linestyle=':', linewidth=0.8)

    ax.set_xlabel(r'$\overline{M^*}_{\rm SSP}$ [kg m$^{-2}$ s$^{-1}$]', fontsize=10)
    ax.set_ylabel(r'$\Delta \overline{M^*}$ (SAI$-$SSP) [kg m$^{-2}$ s$^{-1}$]', fontsize=10)
    ax.set_title(model_name, fontsize=11, fontweight='bold')
    ax.legend(fontsize=10, framealpha=0.9)
    ax.grid(True, alpha=0.3)

for idx in range(len(alpha_store), 6):
    axes[idx].axis('off')

fig.suptitle(r'$\Delta \overline{M^*}$ vs $\overline{M^*}_{\rm SSP}$ (time-mean, grid points 30°S–30°N)'
             f'\nMMM α (= tropmean(dM*) / tropmean(M*_SSP))= {alpha_mmm:.4f}',
             fontsize=13, fontweight='bold')

plt.tight_layout()
# plt.savefig('alpha_scatter_spatial.png', dpi=150, bbox_inches='tight')
plt.show()

/badc/cmip6/data/CMIP6/GeoMIP/MOHC/UKESM1-0-LL/G6sulfur/r1i1p1f2/Amon/pr/gn/latest


HDF5-DIAG: Error detected in HDF5 (1.14.3) thread 0:
  #000: H5A.c line 1891 in H5Aiterate2(): invalid location identifier
    major: Invalid arguments to routine
    minor: Inappropriate type
  #001: H5VLint.c line 1741 in H5VL_vol_object(): invalid identifier type to function
    major: Invalid arguments to routine
    minor: Inappropriate type


RuntimeError: NetCDF: Can't open HDF5 attribute

In [ ]:
from scipy import stats
import matplotlib.colors as mcolors

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (model_name, res) in enumerate(alpha_store.items()):
    ax    = axes[idx]
    alpha = res['alpha']

    x = res['Mstar_ssp'].values.flatten()
    y = res['dMstar'].values.flatten()

    valid   = np.isfinite(x) & np.isfinite(y)
    x_clean = x[valid]
    y_clean = y[valid]

    # --- OLS regression ---
    slope, intercept, r_value, p_value, _ = stats.linregress(x_clean, y_clean)
    r2 = r_value**2

    # --- 2D histogram (density) instead of scatter ---
    h, xedges, yedges = np.histogram2d(x_clean, y_clean, bins=100)
    # mask empty bins
    h = np.ma.masked_where(h == 0, h)

    pcm = ax.pcolormesh(xedges, yedges, h.T,
                        norm=mcolors.LogNorm(vmin=1),
                        cmap='YlOrRd')
    plt.colorbar(pcm, ax=ax, label='Count', pad=0.02)

    # --- Alpha line through origin ---
    x_line = np.linspace(x_clean.min(), x_clean.max(), 200)
    ax.plot(x_line, alpha * x_line,
            color='red', linewidth=2, linestyle='--',
            label=rf'$\alpha={alpha:.4f}$')

    # --- OLS regression line ---
    ax.plot(x_line, slope * x_line + intercept,
            color='blue', linewidth=2, linestyle='-',
            label=rf'OLS (slope={slope:.4f})')

    ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
    ax.axvline(0, color='gray', linestyle=':', linewidth=0.8)

    # --- R² inset text ---
    ax.text(0.97, 0.97, f'$R^2={r2:.3f}$\n$p={p_value:.2e}$',
            transform=ax.transAxes,
            fontsize=9, va='top', ha='right',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

    ax.set_xlabel(r'$\overline{M^*}_{\rm SSP}$ [kg m$^{-2}$ s$^{-1}$]', fontsize=10)
    ax.set_ylabel(r'$\Delta \overline{M^*}$ (SAI$-$SSP) [kg m$^{-2}$ s$^{-1}$]', fontsize=10)
    ax.set_title(model_name, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9, framealpha=0.9, loc='upper left')
    ax.grid(True, alpha=0.2)

for idx in range(len(alpha_store), 6):
    axes[idx].axis('off')

alpha_mmm = np.mean([v['alpha'] for v in alpha_store.values()])
fig.suptitle(r'$\Delta \overline{M^*}$ vs $\overline{M^*}_{\rm SSP}$ (time-mean, grid points 30°S–30°N)'
             f'\nMMM α = {alpha_mmm:.4f}',
             fontsize=13, fontweight='bold')

plt.tight_layout()
# plt.savefig('alpha_scatter_density.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from scipy import stats
import matplotlib.colors as mcolors

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (model_name, res) in enumerate(alpha_store.items()):
    ax    = axes[idx]
    alpha = res['alpha']

    x = res['Mstar_ssp'].values.flatten()
    y = res['dMstar'].values.flatten()

    valid   = np.isfinite(x) & np.isfinite(y)
    x_clean = x[valid]
    y_clean = y[valid]

    # --- OLS regression ---
    slope, intercept, r_value, p_value, _ = stats.linregress(x_clean, y_clean)
    r2 = r_value**2

    # --- 2D histogram (density) instead of scatter ---
    h, xedges, yedges = np.histogram2d(x_clean, y_clean, bins=100)
    # mask empty bins
    h = np.ma.masked_where(h == 0, h)

    cf = ax.contourf(0.5*(xedges[:-1]+xedges[1:]),
                     0.5*(yedges[:-1]+yedges[1:]),
                     h.T,
                     # levels=15,
                     levels=np.linspace(0, 60, 7), extend='max',
                     # norm=mcolors.LogNorm(vmin=1),
                     cmap='YlOrRd')
    plt.colorbar(cf, ax=ax, label='Count', pad=0.02)

    # --- Alpha line through origin ---
    x_line = np.linspace(x_clean.min(), x_clean.max(), 200)
    ax.plot(x_line, alpha * x_line,
            color='red', linewidth=2, linestyle='--',
            label=rf'$\alpha={alpha:.4f}$')

    # --- OLS regression line ---
    ax.plot(x_line, slope * x_line + intercept,
            color='blue', linewidth=2, linestyle='-',
            label=rf'OLS (slope={slope:.4f})')

    ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
    ax.axvline(0, color='gray', linestyle=':', linewidth=0.8)

    # --- R² inset text ---
    ax.text(0.97, 0.97, f'$R^2={r2:.3f}$\n$p={p_value:.2e}$',
            transform=ax.transAxes,
            fontsize=9, va='top', ha='right',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

    ax.set_xlabel(r'$\overline{M^*}_{\rm SSP}$ [kg m$^{-2}$ s$^{-1}$]', fontsize=10)
    ax.set_ylabel(r'$\Delta \overline{M^*}$ (SAI$-$SSP) [kg m$^{-2}$ s$^{-1}$]', fontsize=10)
    ax.set_title(model_name, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9, framealpha=0.9, loc='upper left')
    ax.grid(True, alpha=0.2)
    ax.set_xlim(0, 0.01)
    ax.set_ylim(-0.0005, 0.0005)

for idx in range(len(alpha_store), 6):
    axes[idx].axis('off')

alpha_mmm = np.mean([v['alpha'] for v in alpha_store.values()])
fig.suptitle(r'$\Delta \overline{M^*}$ vs $\overline{M^*}_{\rm SSP}$ (time-mean, grid points 30°S–30°N)'
             f'\nMMM α = {alpha_mmm:.4f}',
             fontsize=13, fontweight='bold')

plt.tight_layout()
# plt.savefig('alpha_scatter_density_II.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from scipy import stats
import matplotlib.colors as mcolors

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (model_name, res) in enumerate(alpha_store.items()):
    ax    = axes[idx]
    alpha = res['alpha']

    x = res['Mstar_ssp'].values.flatten()
    y = res['dMstar'].values.flatten()

    valid   = np.isfinite(x) & np.isfinite(y)
    x_clean = x[valid]
    y_clean = y[valid]

    # --- OLS regression ---
    slope, intercept, r_value, p_value, _ = stats.linregress(x_clean, y_clean)
    r2 = r_value**2

    # --- 2D histogram (density) instead of scatter ---
    h, xedges, yedges = np.histogram2d(x_clean, y_clean, bins=100)
    # mask empty bins
    h = np.ma.masked_where(h == 0, h)

    cf = ax.contourf(0.5*(xedges[:-1]+xedges[1:]),
                     0.5*(yedges[:-1]+yedges[1:]),
                     h.T,
                     # levels=15,
                     levels=np.linspace(0, 60, 7), extend='max',
                     # norm=mcolors.LogNorm(vmin=1),
                     cmap='YlOrRd')
    plt.colorbar(cf, ax=ax, label='Count', pad=0.02)

    # --- Alpha line through origin ---
    x_line = np.linspace(x_clean.min(), x_clean.max(), 200)
    ax.plot(x_line, alpha * x_line,
            color='red', linewidth=2, linestyle='--',
            label=rf'$\alpha={alpha:.4f}$')

    # --- OLS regression line ---
    ax.plot(x_line, slope * x_line + intercept,
            color='blue', linewidth=2, linestyle='-',
            label=rf'Reg (slope={slope:.4f})')

    ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
    ax.axvline(0, color='gray', linestyle=':', linewidth=0.8)

    # --- R² inset text ---
    ax.text(0.97, 0.97, f'$R^2={r2:.3f}$',
            transform=ax.transAxes,
            fontsize=9, va='top', ha='right',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

    ax.set_xlabel(r'$\overline{M^*}_{\rm SSP}$ [kg m$^{-2}$ s$^{-1}$]', fontsize=10)
    ax.set_ylabel(r'$\Delta \overline{M^*}$ (SAI$-$SSP) [kg m$^{-2}$ s$^{-1}$]', fontsize=10)
    ax.set_title(model_name, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9, framealpha=0.9, loc='upper left')
    ax.grid(True, alpha=0.2)
    ax.set_xlim(0, 0.01)
    ax.set_ylim(-0.0005, 0.0005)


# --- outside the loop ---
for idx in range(len(alpha_store) + 1, 6):
    axes[idx].axis('off')

# --- MMM panel ---
ax_mmm = axes[len(alpha_store)]

dMstar_mmm    = np.nanmean([res['dMstar'].values    for res in alpha_store.values()], axis=0)
Mstar_ssp_mmm = np.nanmean([res['Mstar_ssp'].values for res in alpha_store.values()], axis=0)

x_mmm = Mstar_ssp_mmm.flatten()
y_mmm = dMstar_mmm.flatten()

valid_mmm   = np.isfinite(x_mmm) & np.isfinite(y_mmm)
x_mmm_clean = x_mmm[valid_mmm]
y_mmm_clean = y_mmm[valid_mmm]

slope_mmm, intercept_mmm, r_mmm, p_mmm, _ = stats.linregress(x_mmm_clean, y_mmm_clean)
r2_mmm = r_mmm**2

h_mmm, xedges_mmm, yedges_mmm = np.histogram2d(x_mmm_clean, y_mmm_clean, bins=100)
h_mmm = np.ma.masked_where(h_mmm == 0, h_mmm)

cf_mmm = ax_mmm.contourf(0.5*(xedges_mmm[:-1]+xedges_mmm[1:]),
                          0.5*(yedges_mmm[:-1]+yedges_mmm[1:]),
                          h_mmm.T,
                          levels=np.linspace(0, 60, 7), extend='max',
                          cmap='YlOrRd')
plt.colorbar(cf_mmm, ax=ax_mmm, label='Count', pad=0.02)

x_line_mmm = np.linspace(0, 0.01, 200)
ax_mmm.plot(x_line_mmm, alpha_mmm * x_line_mmm,
            color='red', linewidth=2, linestyle='--',
            label=rf'$\alpha_{{MMM}}={alpha_mmm:.4f}$')
ax_mmm.plot(x_line_mmm, slope_mmm * x_line_mmm + intercept_mmm,
            color='blue', linewidth=2, linestyle='-',
            label=rf'Reg (slope={slope_mmm:.4f})')

ax_mmm.axhline(0, color='gray', linestyle=':', linewidth=0.8)
ax_mmm.axvline(0, color='gray', linestyle=':', linewidth=0.8)

ax_mmm.text(0.97, 0.97, f'$R^2={r2_mmm:.3f}$',
            transform=ax_mmm.transAxes,
            fontsize=9, va='top', ha='right',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

ax_mmm.set_xlim(0, 0.01)
ax_mmm.set_ylim(-0.0005, 0.0005)
ax_mmm.set_xlabel(r'$\overline{M^*}_{\rm SSP}$ [kg m$^{-2}$ s$^{-1}$]', fontsize=10)
ax_mmm.set_ylabel(r'$\Delta \overline{M^*}$ (SAI$-$SSP) [kg m$^{-2}$ s$^{-1}$]', fontsize=10)
ax_mmm.set_title('Multi-Model Mean', fontsize=11, fontweight='bold')
ax_mmm.legend(fontsize=9, framealpha=0.9, loc='upper left')
ax_mmm.grid(True, alpha=0.2)

alpha_mmm = np.mean([v['alpha'] for v in alpha_store.values()])
fig.suptitle(r'$\Delta \overline{M^*}$ vs $\overline{M^*}_{\rm SSP}$ (time-mean, grid points 30°S–30°N)'
             f'\nMMM α (= tropmean(dM*) / tropmean(M*_SSP)) = {alpha_mmm:.4f}',
             fontsize=13, fontweight='bold')

plt.tight_layout()
# plt.savefig('alpha_contourf_density_III.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
alpha_mmm

In [ ]:
# ALPHA = -0.284
ALPHA = alpha_mmm

In [ ]:
# ================================================================
# CHADWICK DECOMPOSITION FUNCTIONS
# Based on Chadwick et al. (2013), following DaAllada et al. (2020)
# P = M* * q_s,  M* = P / q_s
# ΔP = M*_ref * Δq_s  +  q_s_ref * ΔM*  +  Δq_s * ΔM*
#    = ΔP_therm       +  ΔP_dyn          +  ΔP_cross
# ΔP_dyn = ΔP_weak + ΔP_shift
# ΔM*_weak  = alpha * M*_ref
# ΔM*_shift = ΔM* - ΔM*_weak
# ================================================================



def compute_Mstar(pr, huss, min_huss=1e-6):
    """M* = P / q_s"""
    huss_safe = huss.where(huss > min_huss, min_huss)
    return pr / huss_safe


def chadwick_decomposition(pr1, pr2, huss1, huss2, alpha=ALPHA):
    """
    Chadwick et al. (2013) decomposition.
    Experiment 1 = G6sulfur, Experiment 2 = SSP245 (reference).

    Parameters
    ----------
    pr1, pr2     : climatological mean precipitation  [kg m-2 s-1]
    huss1, huss2 : climatological mean near-surface q [kg/kg]
    alpha        : scalar, tropical mean ΔM*/M*_ref

    Returns
    -------
    dict of spatial fields [kg m-2 s-1]
    """
    Mstar_ref = compute_Mstar(pr2, huss2)    # M*_SSP245
    qs_ref    = huss2                          # q_s_SSP245
    Mstar_g6  = compute_Mstar(pr1, huss1)

    dqs    = huss1 - huss2                    # Δq_s
    dMstar = Mstar_g6 - Mstar_ref             # ΔM*

    dMstar_weak  = alpha * Mstar_ref           # uniform weakening
    dMstar_shift = dMstar - dMstar_weak        # spatial reorganisation

    dP_therm = Mstar_ref * dqs                # thermodynamic
    dP_dyn   = qs_ref    * dMstar             # total dynamic
    dP_weak  = qs_ref    * dMstar_weak        # dynamic-weak
    dP_shift = qs_ref    * dMstar_shift       # dynamic-shift
    dP_cross = dqs       * dMstar             # nonlinear cross

    dP_total = dP_therm + dP_dyn + dP_cross

    return {
        'Mstar_ref':    Mstar_ref,
        'Mstar_g6':     Mstar_g6,
        'dMstar':       dMstar,
        'dMstar_weak':  dMstar_weak,
        'dMstar_shift': dMstar_shift,
        'dP_therm':     dP_therm,
        'dP_dyn':       dP_dyn,
        'dP_weak':      dP_weak,
        'dP_shift':     dP_shift,
        'dP_cross':     dP_cross,
        'dP_total':     dP_total,
    }


# ================================================================
# MAIN PROCESSING LOOP
# ================================================================

if not os.path.exists(output_file):
    print('#'*40)
    print("File does not exist — running Chadwick decomposition")
    print('#'*40)

    all_results = {}

    for model_name, model_meta in MODELS.items():

        components_by_exp = {}
        seasonal_data     = {}

        # ----------------------------
        # 1) LOAD DATA
        # ----------------------------
        for exp, meta in EXPERIMENTS.items():

            if model_name == "CESM2-WACCM":
                ensemble = "r1i1p1f2" if meta["scenario"] == "G6sulfur" else "r1i1p1f1"
            else:
                ensemble = model_meta["ensemble"]

            def make_base(vname):
                return (
                    CEDA_BASE
                    / meta["project"]
                    / model_meta["institution"]
                    / model_name
                    / meta["scenario"]
                    / ensemble
                    / "Amon"
                    / vname
                    / model_meta["grid"]
                    / "latest"
                )

            base_pr   = make_base("pr")
            base_huss = make_base("huss")

            ds_pr   = load_model_data_alt_path(base_pr,   meta["project"], model_meta["institution"],
                                               model_name, meta["scenario"], ensemble, model_meta["grid"], mf, "pr")
            ds_huss = load_model_data_alt_path(base_huss, meta["project"], model_meta["institution"],
                                               model_name, meta["scenario"], ensemble, model_meta["grid"], mf, "huss")

            ds_pr   = roll_and_regrid(ds_pr)
            ds_huss = roll_and_regrid(ds_huss)

            print(f"  [{model_name}] {exp}: pr loaded {ds_pr.sizes['time']} months")
            print(f"  [{model_name}] {exp}: huss loaded {ds_huss.sizes['time']} months")

            components_by_exp[exp] = {
                'P':    ds_pr['pr'],
                'huss': ds_huss['huss'],
            }

        # ----------------------------
        # 2) SEASONAL PROCESSING
        # ----------------------------
        for exp in components_by_exp:
            seasonal_data[exp] = {}

            t_start, t_end = _time_slice(exp)
            comp = components_by_exp[exp]

            pr_sliced   = comp['P'].sel(time=slice(t_start, t_end))
            huss_sliced = comp['huss'].sel(time=slice(t_start, t_end))

            n_months = pr_sliced.sizes['time']
            print(f"  [{model_name}] {exp}: sliced to {n_months} months = {n_months//12} years")

            for season, (m_start, m_end) in [("SUM", (6, 9)), ("WIN", (11, 2)), ("ANN", (1, 12))]:

                print(f"  [{model_name}] {exp} - {season}: processing...")

                pr_seasonal   = mf.seasonal_mean_by_year_old(pr_sliced,   m_start, m_end)
                huss_seasonal = mf.seasonal_mean_by_year_old(huss_sliced, m_start, m_end)

                seasonal_data[exp][season] = {
                    'pr_clim':   pr_seasonal.mean(dim='year'),
                    'huss_clim': huss_seasonal.mean(dim='year'),
                    'pr_std':    pr_seasonal.std(dim='year'),
                }

        # ----------------------------
        # 3) COMPUTE DELTAS + CHADWICK DECOMPOSITION
        # ----------------------------
        results = {exp: seasonal_data[exp] for exp in seasonal_data}

        EXP1, EXP2 = "G6sulfur", "SSP245"

        if EXP1 in results and EXP2 in results:
            results["delta"] = {}

            for season in ["SUM", "WIN", "ANN"]:
                results["delta"][season] = {}

                # Raw delta P
                results["delta"][season]['dP'] = (
                    results[EXP1][season]['pr_clim'] - results[EXP2][season]['pr_clim']
                )

                # Chadwick decomposition
                print(f"  [{model_name}] Chadwick decomposition — {season}...")
                chad = chadwick_decomposition(
                    pr1   = results[EXP1][season]['pr_clim'],
                    pr2   = results[EXP2][season]['pr_clim'],
                    huss1 = results[EXP1][season]['huss_clim'],
                    huss2 = results[EXP2][season]['huss_clim'],
                    alpha = ALPHA,
                )
                results["delta"][season].update(chad)

            print(f"  [{model_name}] All seasons done")

        all_results[model_name] = results

    # ----------------------------
    # 4) SAVE TO NETCDF
    # ----------------------------
    print("\nCreating netCDF output file...")

    first_model = list(all_results.keys())[0]
    sample_da   = all_results[first_model]["delta"]["ANN"]["dP"]
    lat         = sample_da.lat.values
    lon         = sample_da.lon.values
    models      = list(all_results.keys())

    ds_out = xr.Dataset(
        coords={'lat': lat, 'lon': lon, 'model': models}
    )
    ds_out.attrs['title']       = 'Chadwick et al. (2013) Precipitation Decomposition'
    ds_out.attrs['description'] = 'ΔP (G6sulfur - SSP245) decomposed into thermodynamic, dynamic-weak, dynamic-shift, and cross terms'
    ds_out.attrs['alpha']       = str(ALPHA)
    ds_out.attrs['reference']   = 'Chadwick et al. (2013); DaAllada et al. (2020)'

    season_names = {"SUM": "JJA", "WIN": "DJF", "ANN": "Annual"}

    chad_vars = {
        'dP':           ('Total ΔP (G6−SSP245)',                   'kg m-2 s-1'),
        'dP_therm':     ('Thermodynamic: M*_ref × Δq_s',           'kg m-2 s-1'),
        'dP_dyn':       ('Dynamic total: q_s_ref × ΔM*',           'kg m-2 s-1'),
        'dP_weak':      ('Dynamic-weak: q_s_ref × α × M*_ref',     'kg m-2 s-1'),
        'dP_shift':     ('Dynamic-shift: q_s_ref × ΔM*_shift',     'kg m-2 s-1'),
        'dP_cross':     ('Cross term: Δq_s × ΔM*',                 'kg m-2 s-1'),
        'dP_total':     ('Reconstruction: therm+dyn+cross',         'kg m-2 s-1'),
        'dMstar':       ('ΔM* = M*_G6 − M*_SSP245',               's-1'),
        'dMstar_weak':  ('ΔM*_weak = α × M*_ref',                  's-1'),
        'dMstar_shift': ('ΔM*_shift = ΔM* − ΔM*_weak',            's-1'),
        'Mstar_ref':    ('M*_SSP245 = P_ref / q_s_ref',            's-1'),
        'Mstar_g6':     ('M*_G6 = P_G6 / q_s_G6',                 's-1'),
    }

    for season in ["SUM", "WIN", "ANN"]:
        sname = season_names[season]
        for varkey, (long_name, units) in chad_vars.items():
            arr = np.stack([
                all_results[model]["delta"][season][varkey].values
                for model in models
            ], axis=0)   # (n_models, lat, lon)

            ds_out[f'{varkey}_{season}'] = (
                ['model', 'lat', 'lon'], arr,
                {'long_name': f'{long_name} ({sname})',
                 'units':     units}
            )

    print(f"Saving to {output_file}...")
    ds_out.to_netcdf(output_file)
    print(f"✓ Done! {len(ds_out.data_vars)} variables, {len(models)} models.")

else:
    print('#'*40)
    print(f"{output_file} already exists — skipping computation")
    print('#'*40)

In [ ]:
import numpy as np
import xarray as xr
import cftime
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

ds=xr.open_dataset(output_file)

In [ ]:
print("ds",ds)

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

def plot_map(
    da,
    title=None,
    cmap='coolwarm',
    figsize=(12, 6),
    vmin=None,
    vmax=None,
    add_colorbar=True
):
    """
    Plot an xarray DataArray on a global map using cartopy.

    Parameters
    ----------
    da : xarray.DataArray
        Must have 'lat' and 'lon' coordinates.
    title : str, optional
        Plot title.
    cmap : str
        Colormap.
    figsize : tuple
        Figure size.
    vmin, vmax : float, optional
        Color scale limits.
    add_colorbar : bool
        Whether to draw colorbar.
    """

    # Ensure 2D (remove singleton dims like 'model')
    da = da.squeeze()

    # Fix longitude if needed (0–360 → -180–180)
    if da.lon.max() > 180:
        da = da.assign_coords(
            lon=((da.lon + 180) % 360) - 180
        ).sortby('lon')

    # Ensure latitude is increasing
    if da.lat[0] > da.lat[-1]:
        da = da.sortby('lat')

    # Create figure
    fig = plt.figure(figsize=figsize)
    ax = plt.axes(projection=ccrs.PlateCarree())

    # Plot
    # im = ax.pcolormesh(
    #     da.lon,
    #     da.lat,
    #     da,
    #     transform=ccrs.PlateCarree(),
    #     cmap=cmap,
    #     vmin=vmin,
    #     vmax=vmax
    # )
    im = ax.contourf(
        da.lon, 
        da.lat, 
        da,     
        transform=ccrs.PlateCarree(),
        levels=np.linspace(vmin, vmax, 21),      
        cmap='RdBu_r', extend='both')

    # Features
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)

    # Gridlines with labels
    gl = ax.gridlines(draw_labels=True, linewidth=0.5, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {'size': 10}
    gl.ylabel_style = {'size': 10}

    # Colorbar
    if add_colorbar:
        cbar = plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.05)
        cbar.set_label(da.name if da.name else '')

    # Title
    if title:
        plt.title(title)
    else:
        plt.title(da.name if da.name else '')

    plt.show()

In [ ]:
da = ds['dP_ANN'].sel(model='UKESM1-0-LL')*86400
plot_map(da, vmin=-0.5, vmax=0.5)

In [ ]:
def compute_regional_map(
    pr_comp,
    component_name,
    season="stitched",
    trop_bound=50,
    land_mask=None,
    agreement_threshold=4,
    plot=True,
    save_fig=False,
    fig_path=None,
):
    """
    Plot regional map from Bony decomposition netCDF file.
    
    Parameters
    ----------
    pr_comp : xr.Dataset
        Bony decomposition dataset (e.g., from 'Bony_decomposition_G6sulfur_minus_SSP245.nc')
    component_name : str
        Component to plot. Options:
        - 'LHS_delta_P' : Total precipitation change
        - 'RHS_delta_P_therm' : Thermodynamic component
        - 'RHS_delta_P_dyn' : Dynamic component
        - 'RHS_delta_P_total' : Total reconstructed change
        - 'RHS_therm_delta_E' : Evaporation term
        - 'RHS_therm_omega_bar_times_delta_Gamma_q' : ω̄*ΔΓ_q term
        - 'RHS_therm_delta_H_q' : Horizontal advection term
        - 'RHS_therm_delta_Valpha_q' : Residual vertical advection term
        - 'RHS_dyn_Gamma_q_times_delta_omega_bar' : Dynamic term
        - 'residual' : LHS - RHS residual
    season : str, default 'stitched'
        Season to plot: 
        - 'stitched' : SUM for NH (lat>0), WIN for SH (lat<=0)
        - 'SUM' : JJA only
        - 'WIN' : DJF only
        - 'ANN' : Annual mean
    trop_bound : int, default 50
        Latitude boundary (not used when reading from netCDF, but kept for compatibility)
    land_mask : optional
        Land mask (not currently used)
    agreement_threshold : int, default 3
        Number of models that must agree on sign for stippling
    plot : bool, default True
        Whether to create the plot
    save_fig : bool, default False
        Whether to save the figure
    fig_path : str, optional
        Path to save figure
    
    Returns
    -------
    delta_mean : xr.DataArray
        Multi-model mean of the selected component
    stipple_mask : np.ndarray
        Boolean mask for model agreement stippling
    """
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker
    import cartopy.io.shapereader as shpreader
    import xarray as xr
    import warnings
    from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
    import matplotlib.patches as mpatches
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature

    # ------------------------------------------------------------------ #
    # Handle stitched vs single season
    # ------------------------------------------------------------------ #
    if season.lower() == "stitched":
        # Load both SUM and WIN
        var_name_sum = f"{component_name}_SUM"
        var_name_win = f"{component_name}_WIN"
        
        if var_name_sum not in pr_comp or var_name_win not in pr_comp:
            raise ValueError(
                f"Variables '{var_name_sum}' or '{var_name_win}' not found in dataset. "
                f"Available variables: {list(pr_comp.data_vars)}"
            )
        
        data_sum = pr_comp[var_name_sum]  # (model, lat, lon)
        data_win = pr_comp[var_name_win]  # (model, lat, lon)
        
        # Stitch: NH (lat > 0) = SUM, SH (lat <= 0) = WIN
        lat = data_sum.lat
        # Apply stitching - this preserves (model, lat, lon) dimension order
        stacked = xr.where(lat > 0, data_sum, data_win)
        
        season_label = "NH: JJA, SH: DJF"
        
    else:
        # Single season
        var_name = f"{component_name}_{season}"
        
        if var_name not in pr_comp:
            raise ValueError(
                f"Variable '{var_name}' not found in dataset. "
                f"Available variables: {list(pr_comp.data_vars)}"
            )
        
        stacked = pr_comp[var_name]  # (model, lat, lon)
        
        season_labels = {'SUM': 'JJA', 'WIN': 'DJF', 'ANN': 'Annual'}
        season_label = season_labels.get(season, season)
    
    # Ensure dimension order is (model, lat, lon)
    if 'model' in stacked.dims:
        stacked = stacked.transpose('model', 'lat', 'lon')
    
    # ------------------------------------------------------------------ #
    # Multi-model mean and stippling mask
    # ------------------------------------------------------------------ #
    delta_mean = stacked.mean(dim="model").squeeze()
    
    sign_mean = np.sign(delta_mean.values)
    sign_models = np.sign(stacked.values)
    
    # sign_models shape: (model, lat, lon)
    # sign_mean shape: (lat, lon)
    # Need to broadcast: (model, lat, lon) == (1, lat, lon)
    n_agree = np.sum(sign_models == sign_mean[np.newaxis, :, :], axis=0)
    stipple_mask = n_agree >= agreement_threshold
    
    n_models = stacked.sizes['model']
    
    lat = delta_mean.lat.values
    lon = delta_mean.lon.values
    data = delta_mean.values
    LON, LAT = np.meshgrid(lon, lat)
    
    # Convert from kg m-2 s-1 to mm/day
    data_mm_day = data * 86400
    
    # ------------------------------------------------------------------ #
    # Plot
    # ------------------------------------------------------------------ #
    if plot:
        # vmax = float(np.nanpercentile(np.abs(data_mm_day), 98))
        vmax=0.5
        vmin = -vmax
        levels = np.linspace(-vmax, vmax, 21)
        
        fig, ax = plt.subplots(figsize=(10, 5),
                               subplot_kw={"projection": ccrs.PlateCarree()})
        ax.add_feature(cfeature.COASTLINE, linewidth=0.3)
        ax.set_global()
        
        im = ax.contourf(LON, LAT, data_mm_day, levels=levels, cmap='RdBu_r',
                         extend='both', transform=ccrs.PlateCarree())
        
        # Stippling
        stip_lat_flat = LAT[stipple_mask].ravel()[::2]
        stip_lon_flat = LON[stipple_mask].ravel()[::2]
        ax.scatter(stip_lon_flat, stip_lat_flat,
                   s=1.5, color="black", alpha=0.5, zorder=5)
        
        # Coastlines
        try:
            shpfilename = shpreader.natural_earth(
                resolution="110m", category="physical", name="coastline"
            )
            reader = shpreader.Reader(shpfilename)
            for geom in reader.geometries():
                if hasattr(geom, "geoms"):
                    for g in geom.geoms:
                        ax.plot(*g.xy, color="black", linewidth=0.5, zorder=4)
                else:
                    ax.plot(*geom.xy, color="black", linewidth=0.5, zorder=4)
        except Exception:
            pass
        
        cbar = plt.colorbar(im, ax=ax, orientation="horizontal", 
                            pad=0.05, shrink=1.0, aspect=40)
        
        # Get component label for plot
        component_labels = {
            'LHS_delta_P': 'Total ΔP',
            'RHS_delta_P_therm': 'Thermodynamic Component',
            'RHS_delta_P_dyn': 'Dynamic Component ($\Gamma_q \cdot \Delta \overline{\omega}$)',
            'RHS_delta_P_total': 'Total Reconstructed ΔP',
            'RHS_therm_delta_E': 'ΔE (Evaporation)',
            'RHS_therm_omega_bar_times_delta_Gamma_q': '$\overline{\omega} \cdot \Delta \Gamma_q$',
            'RHS_therm_delta_H_q': '$\Delta H_q$ (Horiz. Advection)',
            'RHS_therm_delta_Valpha_q': '$\Delta V^α_q$ (Resid. Vert. Advection)',
            'RHS_dyn_Gamma_q_times_delta_omega_bar': 'Γ_q·Δω̄', #Not needed, it is the dyn component
            'residual': 'Residual (LHS - RHS)'
        }
        
        comp_label = component_labels.get(component_name, component_name)
        
        cbar.set_label(f"{comp_label}  [mm day⁻¹]", fontsize=10)
        
        # Equator line
        ax.plot([float(lon.min()), float(lon.max())], [0, 0],
                color='gray', linewidth=0.8, linestyle='--',
                transform=ccrs.PlateCarree())
        
        ax.set_extent([float(lon.min()), float(lon.max()),
                       float(lat.min()), float(lat.max())], crs=ccrs.PlateCarree())
        ax.set_aspect('auto')
        
        ax.set_xticks(np.arange(np.floor(float(lon.min())),
                                np.ceil(float(lon.max()))+1, 30), crs=ccrs.PlateCarree())
        ax.set_yticks(np.arange(np.floor(float(lat.min())),
                                np.ceil(float(lat.max()))+1, 10), crs=ccrs.PlateCarree())
        ax.xaxis.set_major_formatter(LongitudeFormatter())
        ax.yaxis.set_major_formatter(LatitudeFormatter())
        ax.tick_params(labelsize=8)
        
        ax.set_xlim(lon.min(), lon.max())
        ax.set_ylim(lat.min(), lat.max())
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(
            lambda x, _: f"{int(x)}°E" if x >= 0 else f"{int(-x)}°W"))
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(
            lambda y, _: f"{int(y)}°N" if y >= 0 else f"{int(-y)}°S"))
        ax.set_title(
            f"Multi-model mean: {comp_label}  ({season_label})\n"
            f"G6sulfur - SSP245  |  "
            f"stippling = {agreement_threshold}/{n_models} model sign agreement",
            fontsize=11
        )
        ax.grid(True, linewidth=0.3, alpha=0.4)
        
        # Bombardi and Boos 2021 regions
        regions = {
            'N. America':   {'lon': (248, 258),  'lat': (18,    33)},
            'S. America':   {'lon': (300, 318),  'lat': (-18.5, -7)},
            'Sahel':        {'lon': (0,   40),   'lat': (7,   18.5)},
            'S. Africa':    {'lon': (19.5, 31),  'lat': (-18.5, -7)},
            'India':        {'lon': (73,  87),   'lat': (15,    27)},
            'N. Australia': {'lon': (120.5, 149),'lat': (-21,  -9.5)},
        }
        for name, coords in regions.items():
            lon0, lon1 = coords['lon']
            lat0, lat1 = coords['lat']
            rect = mpatches.Rectangle(
                (lon0, lat0), lon1-lon0, lat1-lat0,
                linewidth=1.2, edgecolor='black', facecolor='none',
                transform=ccrs.PlateCarree()
            )
            ax.add_patch(rect)
            ax.text(lon0+(lon1-lon0)/2, lat0-3, name,
                    fontsize=8, ha='center', va='top',
                    transform=ccrs.PlateCarree())
        
        plt.tight_layout()
        plt.show()
        
        if save_fig:
            season_suffix = "stitched" if season.lower() == "stitched" else season
            path = fig_path or f"{component_name}_{season_suffix}_G6sulfur_minus_SSP245.png"
            fig.savefig(path, dpi=150, bbox_inches="tight")
            print(f"Figure saved → {path}")
    
    season_suffix = "stitched" if season.lower() == "stitched" else season
    # print(f"Returning → {component_name}_{season_suffix}")
    return delta_mean, stipple_mask

In [ ]:
ds

In [ ]:
# Plot total precipitation change (annual)
# Plot stitched map (default): SUM for NH, WIN for SH
delta_mean, stipple = compute_regional_map(
    ds, 
    component_name='dP',
    # season='stitched'  # This is the default
)

In [ ]:
# Plot total precipitation change (annual)
# Plot stitched map (default): SUM for NH, WIN for SH
delta_mean, stipple = compute_regional_map(
    ds, 
    component_name='dP_therm',
    # season='stitched'  # This is the default
)

In [ ]:
# Plot total precipitation change (annual)
# Plot stitched map (default): SUM for NH, WIN for SH
delta_mean, stipple = compute_regional_map(
    ds, 
    component_name='dP_dyn',
    # season='stitched'  # This is the default
)

In [ ]:
# Plot total precipitation change (annual)
# Plot stitched map (default): SUM for NH, WIN for SH
delta_mean, stipple = compute_regional_map(
    ds, 
    component_name='dP_weak',
    trop_bound=30,
    # season='stitched'  # This is the default
)

In [ ]:
# Save
delta_mean.to_netcdf('/gws/ssde/j25b/impose/bidyut/analysis_transient_data/Rainfall_response_RCweak.nc')

In [ ]:
# Plot total precipitation change (annual)
# Plot stitched map (default): SUM for NH, WIN for SH
delta_mean, stipple = compute_regional_map(
    ds, 
    component_name='dP_shift',
    # season='stitched'  # This is the default
)

In [ ]:
# Plot total precipitation change (annual)
# Plot stitched map (default): SUM for NH, WIN for SH
delta_mean, stipple = compute_regional_map(
    ds, 
    component_name='dP_cross',
    # season='stitched'  # This is the default
)

In [ ]:
# Plot total precipitation change (annual)
# Plot stitched map (default): SUM for NH, WIN for SH
delta_mean, stipple = compute_regional_map(
    ds, 
    component_name='dMstar_shift',
    # season='stitched'  # This is the default
)

# Plot bars

In [ ]:
def plot_regional_decomposition_bars(
    pr_comp,
    set_1,
    set_2,
    season="stitched",
    plot_type="percentage",  # 'percentage' or 'absolute'
    show_error_bars=True,
    agreement_threshold=4,
    figsize=(12, 6),
    save_fig=False,
    fig_path=None,
):
    """
    Plot bar chart showing decomposition components for monsoon regions.
    
    Parameters
    ----------
    pr_comp : xr.Dataset
        Bony decomposition dataset
    set_1 : list of str
        Component(s) to use as denominator/total (e.g., ['LHS_delta_P'])
    set_2 : list of str
        Component(s) to decompose (e.g., ['RHS_delta_P_therm', 'RHS_delta_P_dyn'])
    season : str, default 'stitched'
        'stitched', 'SUM', 'WIN', or 'ANN'
    plot_type : str, default 'percentage'
        'percentage' (% contribution) or 'absolute' (mm/day)
    show_error_bars : bool, default True
        Whether to show model spread as error bars
    agreement_threshold : int, default 4
        Minimum models for agreement (not used in bars, kept for compatibility)
    figsize : tuple, default (12, 6)
        Figure size
    save_fig : bool, default False
        Whether to save figure
    fig_path : str, optional
        Path to save figure
    
    Returns
    -------
    fig, ax : matplotlib figure and axes
    regional_data : dict
        Dictionary with regional mean values and spreads
    """
    import numpy as np
    import matplotlib.pyplot as plt
    import xarray as xr
    
    # Bombardi and Boos 2021 regions
    regions = {
        'N. America':   {'lon': (-112, -102),  'lat': (18,    33)},
        'S. America':   {'lon': (-60,  -42),   'lat': (-18.5, -7)},
        'Sahel':        {'lon': (0,   40),   'lat': (7,   18.5)},
        'S. Africa':    {'lon': (19.5, 31),  'lat': (-18.5, -7)},
        'India':        {'lon': (73,  87),   'lat': (15,    27)},
        'N. Australia': {'lon': (120.5, 149),'lat': (-21,  -9.5)},
    }
    
    component_labels = {
        'LHS_delta_P': 'Total ΔP',
        'RHS_delta_P_therm': 'Thermodynamic',
        'RHS_delta_P_dyn': 'Dynamic',
        'RHS_delta_P_total': 'Reconstructed ΔP',
        'RHS_therm_delta_E': 'ΔE',
        'RHS_therm_omega_bar_times_delta_Gamma_q': r'$\overline{\omega} \cdot \Delta \Gamma_q$',
        'RHS_therm_delta_H_q': r'$\Delta H_q$',
        'RHS_therm_delta_Valpha_q': r'$\Delta V^{\alpha}_q$',
        'RHS_dyn_Gamma_q_times_delta_omega_bar': r'$\Gamma_q \cdot \Delta\overline{\omega}$',
        'residual': 'Residual'
    }
    
    # ------------------------------------------------------------------ #
    # Extract data for each component
    # ------------------------------------------------------------------ #
    def get_component_data(component_name, season):
        """Get stitched or single season data for a component."""
        if season.lower() == "stitched":
            var_name_sum = f"{component_name}_SUM"
            var_name_win = f"{component_name}_WIN"
            
            data_sum = pr_comp[var_name_sum]
            data_win = pr_comp[var_name_win]
            
            lat = data_sum.lat
            stacked = xr.where(lat > 0, data_sum, data_win)
        else:
            var_name = f"{component_name}_{season}"
            stacked = pr_comp[var_name]
        
        if 'model' in stacked.dims:
            stacked = stacked.transpose('model', 'lat', 'lon')
        
        return stacked * 86400  # Convert to mm/day
    
    # ------------------------------------------------------------------ #
    # Compute regional means
    # ------------------------------------------------------------------ #
    regional_data = {}
    
    for region_name, coords in regions.items():
        regional_data[region_name] = {}
        
        lon0, lon1 = coords['lon']
        lat0, lat1 = coords['lat']
        
        # Handle longitude wrapping (e.g., 248-258 crosses 360->0)
        for comp_name in set_1 + set_2:
            data = get_component_data(comp_name, season)
            
            # Select region (simple case, no wrapping needed)
            mask_lon = (data.lon >= lon0) & (data.lon <= lon1)
            mask_lat = (data.lat >= lat0) & (data.lat <= lat1)
            
            # Select and compute regional mean for each model
            regional = data.where(mask_lat & mask_lon, drop=True)
            regional_mean_per_model = regional.mean(dim=['lat', 'lon'])
            
            # Multi-model mean and std
            mean_val = regional_mean_per_model.mean(dim='model').values
            std_val = regional_mean_per_model.std(dim='model').values
            
            regional_data[region_name][comp_name] = {
                'mean': mean_val,
                'std': std_val
            }
    
    # ------------------------------------------------------------------ #
    # Compute percentages if needed
    # ------------------------------------------------------------------ #
    if plot_type == "percentage":
        # Compute percentages WHILE PRESERVING SIGN RELATIVE TO ABSOLUTE VALUES
        for region_name in regional_data:
            total = sum(regional_data[region_name][comp]['mean'] for comp in set_1)
            
            for comp in set_2:
                if abs(total) > 1e-10:  # Avoid division by zero
                    # Use abs(total) to preserve the original sign direction
                    regional_data[region_name][comp]['percentage'] = (
                        regional_data[region_name][comp]['mean'] / abs(total) * 100
                    )
                    # Error propagation
                    regional_data[region_name][comp]['percentage_std'] = (
                        regional_data[region_name][comp]['std'] / abs(total) * 100
                    )
                else:
                    regional_data[region_name][comp]['percentage'] = 0
                    regional_data[region_name][comp]['percentage_std'] = 0
    
    # ------------------------------------------------------------------ #
    # Plot
    # ------------------------------------------------------------------ #
    fig, ax = plt.subplots(figsize=figsize)
    
    region_names = list(regions.keys())
    x = np.arange(len(region_names))
    width = 0.8 / len(set_2)
    
    colors = plt.cm.Set2(np.linspace(0, 1, len(set_2)))
    
    if plot_type == "percentage":
        # Separate positive and negative contributions
        pos_bottom = np.zeros(len(region_names))
        neg_bottom = np.zeros(len(region_names))
        
        for i, comp in enumerate(set_2):
            values = np.array([regional_data[reg][comp]['percentage'] for reg in region_names])
            errors = [regional_data[reg][comp]['percentage_std'] for reg in region_names] if show_error_bars else None
            
            label = component_labels.get(comp, comp)
            
            # Determine if positive or negative
            pos_values = np.where(values >= 0, values, 0)
            neg_values = np.where(values < 0, values, 0)
            
            # Plot positive contributions stacked upward
            if np.any(pos_values > 0):
                ax.bar(x, pos_values, width=0.6, bottom=pos_bottom, 
                       label=label, color=colors[i], alpha=0.8,
                       edgecolor='black', linewidth=0.5)
                pos_bottom += pos_values
            
            # Plot negative contributions stacked downward
            if np.any(neg_values < 0):
                ax.bar(x, neg_values, width=0.6, bottom=neg_bottom,
                       color=colors[i], alpha=0.8,
                       edgecolor='black', linewidth=0.5)
                neg_bottom += neg_values
            
            if show_error_bars and errors is not None:
                ax.errorbar(x, values, yerr=errors, fmt='none', 
                           ecolor='black', elinewidth=1, capsize=3, alpha=0.6)
        
        ax.set_ylabel('Contribution (%)', fontsize=12)
        ax.axhline(y=100, color='gray', linestyle='--', linewidth=1, alpha=0.5)
        ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
        
        # Add reference line name
        total_label = ' + '.join([component_labels.get(c, c) for c in set_1])
        ax.text(0.02, 0.98, f'Relative to: {total_label}', 
               transform=ax.transAxes, fontsize=9, va='top',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
        
    else:  # absolute
        # Grouped bars for absolute values
        for i, comp in enumerate(set_2):
            values = [regional_data[reg][comp]['mean'] for reg in region_names]
            errors = [regional_data[reg][comp]['std'] for reg in region_names] if show_error_bars else None
            
            label = component_labels.get(comp, comp)
            offset = (i - len(set_2)/2 + 0.5) * width
            
            bars = ax.bar(x + offset, values, width, 
                         label=label, color=colors[i], alpha=0.8,
                         edgecolor='black', linewidth=0.5,
                         yerr=errors if show_error_bars else None,
                         capsize=3, error_kw={'elinewidth': 1, 'alpha': 0.6})
        
        ax.set_ylabel('ΔP (mm day⁻¹)', fontsize=12)
        ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
    
    ax.set_xlabel('Monsoon Region', fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(region_names, rotation=15, ha='right')
    ax.legend(loc='best', fontsize=10, framealpha=0.9)
    ax.grid(True, alpha=0.3, axis='y')
    
    season_labels = {'stitched': 'NH: JJA, SH: DJF', 'SUM': 'JJA', 'WIN': 'DJF', 'ANN': 'Annual'}
    season_label = season_labels.get(season.lower(), season)
    
    title = f"Precipitation Decomposition by Region ({season_label})\n"
    if plot_type == "percentage":
        title += f"Contribution to: {total_label}"
    else:
        title += "Absolute Changes (G6sulfur - SSP585)"
    
    ax.set_title(title, fontsize=13, fontweight='bold')
    
    plt.tight_layout()
    
    if save_fig:
        season_suffix = "stitched" if season.lower() == "stitched" else season
        path = fig_path or f"regional_decomposition_{plot_type}_{season_suffix}.png"
        fig.savefig(path, dpi=300, bbox_inches='tight')
        print(f"Figure saved → {path}")
    
    plt.show()
    
    return fig, ax, regional_data

In [ ]:
# Plot
fig, ax, data = plot_regional_decomposition_bars(
    ds,
    set_1=['dP'],
    set_2=['dP_therm', 'dP_cross', 'dP_dyn'],
    season='stitched',
    plot_type='absolute'
)

In [ ]:
# Plot
fig, ax, data = plot_regional_decomposition_bars(
    ds,
    set_1=['dP'],
    set_2=['dP_therm', 'dP_cross', 'dP_dyn'],
    season='stitched',
    plot_type='percentage'
)

In [ ]:
# Plot
fig, ax, data = plot_regional_decomposition_bars(
    ds,
    set_1=['dP_dyn'],
    set_2=['dP_shift', 'dP_weak'],
    season='stitched',
    plot_type='absolute'
)